# Étape 3 — Contrôle qualité

Compare `OCR.txt` et `EDIT.txt` pour détecter ce que l'édition aurait perdu, et produit `<Livre>_REPORT.txt`.

**Le texte n'est jamais modifié.** Cette étape produit un diagnostic, pas une correction : c'est à vous de décider quoi faire de ce qu'elle signale.

---

**Ce notebook n'est qu'une interface.** Toute la logique vit dans le paquet
`theatre_editor`. On y monte le Drive, on installe les dépendances, on surcharge
éventuellement la configuration, puis on lance l'étape.

**Cette étape est reprenable.** Si Colab coupe, relancez la cellule
d'exécution : le travail déjà validé ne sera pas refait, et vous ne repaierez
aucun appel.

## 1. Dépendances et montage du Drive

In [ ]:
# Installation des dépendances du pipeline.
!pip install -q -U openai pymupdf python-docx

from google.colab import drive

drive.mount("/content/drive")

## 2. Récupération du code

Le dépôt [`elyeskaak/texte_troupe_theatre`](https://github.com/elyeskaak/texte_troupe_theatre)
est **public** : rien à configurer, la cellule suivante suffit. Elle récupère la
dernière version du code à chaque exécution.

> **Si vous repassiez le dépôt en privé**, il faudrait un jeton d'accès :
> GitHub → *Settings* → *Developer settings* → *Personal access tokens* →
> *Fine-grained tokens*, avec **Contents : Read-only** sur ce seul dépôt. Puis
> l'enregistrer dans les Secrets de Colab sous le nom `GITHUB_TOKEN`. La
> cellule le détecte et l'utilise automatiquement — aucune modification à faire.

In [ ]:
# --- Option A : récupération depuis GitHub ------------------------------
DEPOT_COMPTE = "elyeskaak"
DEPOT_NOM = "texte_troupe_theatre"
DOSSIER_PROJET = f"/content/{DEPOT_NOM}"

import os
import subprocess
import sys

# Le jeton est OPTIONNEL : inutile sur un dépôt public, utilisé
# automatiquement s'il est présent. Ainsi la cellule fonctionne dans les deux
# cas, sans qu'il faille se souvenir de la visibilité du dépôt.
jeton = None

try:
    from google.colab import userdata

    jeton = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

# Quand un jeton est utilisé, l'URL le contient : elle ne doit JAMAIS être
# affichée ni figurer dans un message d'erreur. Les sorties de git sont donc
# capturées, jamais relayées.
if jeton:
    url = f"https://{jeton}@github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"
else:
    url = f"https://github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"

if os.path.isdir(DOSSIER_PROJET):
    commande = ["git", "-C", DOSSIER_PROJET, "pull", "--quiet"]
else:
    commande = ["git", "clone", "--quiet", url, DOSSIER_PROJET]

resultat = subprocess.run(commande, capture_output=True, text=True)

if resultat.returncode != 0:
    raise RuntimeError(
        "Récupération du code impossible.\n"
        f"Vérifiez que le dépôt {DEPOT_COMPTE}/{DEPOT_NOM} est accessible.\n"
        "S'il est privé, ajoutez un secret GITHUB_TOKEN dans Colab."
    )

if DOSSIER_PROJET not in sys.path:
    sys.path.insert(0, DOSSIER_PROJET)

print("Code récupéré :", DOSSIER_PROJET)
print("Jeton GitHub  :", "utilisé" if jeton else "non nécessaire (dépôt public)")

In [ ]:
# --- Option B : le dossier theatre_editor/ est sur votre Drive ---------
# Décommentez ces lignes et ajustez le chemin, puis n'exécutez PAS l'option A.

# import sys
# DOSSIER_PROJET = "/content/drive/MyDrive/texte_troupe_theatre"
# if DOSSIER_PROJET not in sys.path:
#     sys.path.insert(0, DOSSIER_PROJET)

## 3. Configuration

`config.py` porte toutes les valeurs par défaut. Les surcharges ci-dessous ne
valent que pour cette session : elles ne modifient pas le fichier.

**Vérifiez le dossier de travail** avant de continuer.

In [ ]:
from pathlib import Path

from theatre_editor import config

# Dossier Drive contenant les PDF et recevant toutes les sorties.
config.DOSSIER_DRIVE = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")

print("Dossier de travail :", config.DOSSIER_DRIVE)
print("Existe             :", config.DOSSIER_DRIVE.is_dir())

## 4. Clé API

In [ ]:
# La clé API est lue depuis les Secrets de Colab.
#
#   panneau latéral « 🔑 Secrets » → ajouter OPENAI_API_KEY
#   → activer « Accès au notebook »
#
# Ainsi la clé n'apparaît jamais dans le notebook ni dans ses sorties.

from theatre_editor.utils import io

try:
    io.charger_cle_api()
    print("Clé API trouvée.")
except RuntimeError as erreur:
    print(erreur)

## 5. Contrôles mécaniques d'abord

Ces contrôles sont **gratuits et instantanés** : aucun appel API. Lancez-les
seuls pour un premier avis, avant d'engager la comparaison par le modèle.

In [ ]:
from theatre_editor.utils import blocks, io

for chemin in io.lister_fichiers_ocr(config.DOSSIER_DRIVE):
    nom = io.nom_livre_depuis_ocr(chemin)
    chemins = io.resoudre_chemins(nom, chemin.parent)

    if not chemins.edit.exists():
        print(f"{nom} : pas encore édité.")
        continue

    constats = blocks.controles_mecaniques(
        io.lire_texte(chemin), io.lire_texte(chemins.edit)
    )

    print(f"{nom} : {len(constats)} constat(s) mécanique(s)")
    for constat in constats:
        print("   ", constat)

## 6. Lancement de la comparaison complète

Un appel par bloc. Reprenable.

In [ ]:
from theatre_editor import validation

resultats = validation.executer(config.DOSSIER_DRIVE)

## 7. Lecture du rapport

Le rapport est fait pour être lu par un humain. Il ne détaille que les blocs
porteurs de constats.

In [ ]:
for resultat in resultats:
    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)

    if chemins.report.exists():
        print(io.lire_texte(chemins.report))
    else:
        print(f"{resultat.nom} : aucun rapport produit.")

## 8. Que faire d'un constat ?

Le rapport signale, il ne corrige pas. Trois façons d'agir, de la plus légère à
la plus lourde.

**Corriger `EDIT.txt` à la main.** Le plus simple pour quelques constats
isolés. Le fichier est du texte, ouvrez-le et corrigez. Relancez ensuite
l'étape 4 seule.

**Refaire un bloc.** Supprimez ses fichiers dans `_EDIT_blocs/` et
`_EDIT_raccords/`, puis relancez l'étape 2 : seul ce bloc sera repayé.

**Réviser un prompt.** Si le même défaut revient sur beaucoup de blocs, c'est
le prompt qu'il faut corriger, dans `theatre_editor/prompts/`. Supprimez alors
`_EDIT_blocs/` en entier pour refaire le livre — et ne changez jamais de prompt
au milieu d'un livre, cela produirait des blocs hétérogènes.

In [ ]:
# Exemple : refaire le bloc 12 d'un livre.
# Décommentez et ajustez le nom et le numéro.

# NOM_LIVRE = "Le Malentendu"
# NUMERO = 12
#
# chemins = io.resoudre_chemins(NOM_LIVRE, config.DOSSIER_DRIVE)
#
# for chemin in (
#     chemins.bloc_txt(NUMERO), chemins.bloc_json(NUMERO),
#     chemins.raccord_txt(NUMERO), chemins.raccord_json(NUMERO),
#     chemins.report_bloc_txt(NUMERO), chemins.report_bloc_json(NUMERO),
# ):
#     if chemin.exists():
#         chemin.unlink()
#         print("supprimé :", chemin.name)

## 9. Journal

In [ ]:
# Journal détaillé de l'étape : un enregistrement par appel API, avec sa
# date, son modèle, son response_id, sa durée et sa consommation de jetons.

import json

chemin = config.DOSSIER_DRIVE / config.NOM_JOURNAL.format(etape="validation")

if chemin.exists():
    journal = json.loads(chemin.read_text(encoding="utf-8"))
    print("Dernière exécution :", journal["derniere_execution"])
    print("Configuration      :", json.dumps(journal["configuration"], ensure_ascii=False))
    print()

    for nom, bilan in journal["livres"].items():
        print(f"{nom} : {json.dumps(bilan, ensure_ascii=False)}")

    jetons = sum(
        (appel.get("tokens_entree") or 0) + (appel.get("tokens_sortie") or 0)
        for appel in journal["appels"]
    )
    print()
    print(f"{len(journal['appels'])} appel(s) journalisé(s), {jetons:,} jetons".replace(",", " "))
else:
    print("Aucun journal : l'étape n'a pas encore été lancée.")